In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import torch

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
assert PROJECT.is_dir(), f"项目目录不存在：{PROJECT}"

print("Python:", platform.python_version())
for package in (
    "torch", "torchvision", "numpy", "Pillow",
    "scikit-learn", "facenet-pytorch",
):
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

selected = {
    "r18_seed0": ("epoch_03.pt", "models/resnet18.py"),
    "r50_seed0": ("epoch_03.pt", "models/resnet50.py"),
    "m3_seed0": ("epoch_02.pt", "models/resnet18_geometry.py"),
}

for run, (checkpoint_name, model_relative) in selected.items():
    run_dir = PROJECT / "runs" / run
    print(f"\n===== {run} =====")
    print("Run directory exists:", run_dir.is_dir())
    print("Selected checkpoint:", run_dir / checkpoint_name)
    print("Checkpoint exists:", (run_dir / checkpoint_name).is_file())

    config_path = run_dir / "config.json"
    print("Training config exists:", config_path.is_file())
    if config_path.is_file():
        config = json.loads(config_path.read_text(encoding="utf-8"))
        print("Training config:")
        print(json.dumps(config, indent=2, ensure_ascii=False))

    source_dir = run_dir / "source"
    print("Source snapshot exists:", source_dir.is_dir())
    if source_dir.is_dir():
        print("Snapshot files:")
        for path in sorted(source_dir.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts:
                print(" ", path.relative_to(source_dir))

    model_path = source_dir / model_relative
    print("Expected model source:", model_path)
    print("Model source exists:", model_path.is_file())
    if model_path.is_file():
        data = model_path.read_bytes()
        print("Model source SHA256:", hashlib.sha256(data).hexdigest())
        print("Model source:")
        print(data.decode("utf-8"))

print("\nDay 19 environment and training-source inspection: COMPLETE")

Mounted at /content/drive
Python: 3.13.15
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
numpy: 2.1.3
Pillow: 11.3.0
scikit-learn: 1.6.1
facenet-pytorch: NOT INSTALLED
CUDA available: True
GPU: Tesla T4

===== r18_seed0 =====
Run directory exists: True
Selected checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/r18_seed0/epoch_03.pt
Checkpoint exists: True
Training config exists: True
Training config:
{
  "seed": 0,
  "epochs": 10,
  "batch_size": 64,
  "num_workers": 2,
  "optimizer": "Adam",
  "learning_rate": 0.0001,
  "weight_decay": 0.0,
  "precision": "FP32",
  "initial_weights": "ResNet18_Weights.IMAGENET1K_V1",
  "augmentation": "None; use existing shared preprocessing",
  "f1_threshold": 0.5,
  "selection_metric": "validation_mAP",
  "attribute_names": [
    "Smiling",
    "Mouth_Slightly_Open",
    "Eyeglasses",
    "Bangs",
    "Wearing_Hat",
    "Mustache",
    "No_Beard",
    "Black_Hair",
    "Bushy_Eyebrows",
    "Arched_Eyebrows",
    "High_Cheekbones",
    "Bag

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import importlib.util
import json
import zipfile
import torch

uploaded = files.upload()
assert len(uploaded) == 1, "请只上传 day19_frozen_inputs.zip"
zip_name = next(iter(uploaded))
WORKSPACE = Path("/content/day19_benchmark_workspace")

def sha(data):
    return hashlib.sha256(data).hexdigest()

def sha_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# 验证整个包，再写入独立工作目录。
with zipfile.ZipFile(zip_name) as archive:
    manifest_name = "day19_handoff_manifest.json"
    handoff = json.loads(archive.read(manifest_name))
    expected_names = set(handoff["files"]) | {manifest_name}
    assert len(archive.namelist()) == len(expected_names)
    assert set(archive.namelist()) == expected_names

    payloads = {}
    for name, expected_hash in handoff["files"].items():
        relative = Path(name)
        assert not relative.is_absolute() and ".." not in relative.parts
        data = archive.read(name)
        assert sha(data) == expected_hash, f"ZIP file changed: {name}"
        payloads[name] = data

    payloads[manifest_name] = archive.read(manifest_name)

for name, data in payloads.items():
    destination = WORKSPACE / name
    if destination.exists():
        assert destination.read_bytes() == data, (
            f"Existing workspace file differs; preserved: {name}"
        )

for name, data in payloads.items():
    destination = WORKSPACE / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)

freeze = json.loads(
    (WORKSPACE / "docs/results/benchmark_freeze/day17/freeze_audit.json")
    .read_text()
)
assert freeze["passed"] is True
for name, expected_hash in freeze["source_and_artifact_sha256"].items():
    assert sha_file(WORKSPACE / name) == expected_hash, name

BENCHMARK = json.loads(
    (WORKSPACE / "configs/benchmark_v1.json").read_text()
)
assert BENCHMARK["test_sample_count"] == 19962
print("Frozen ZIP and Day 17 audit: PASS")

model_specs = {
    "r18_seed0": ("models/resnet18.py", "ResNet18Attributes"),
    "r50_seed0": ("models/resnet50.py", "ResNet50Attributes"),
    "m3_seed0": (
        "models/resnet18_geometry.py", "ResNet18GeometryAttributes"
    ),
}
MODELS = {}
MODEL_PROVENANCE = {}

for run, (relative, class_name) in model_specs.items():
    print(f"\n===== {run} =====")
    run_dir = PROJECT / "runs" / run
    training_config = json.loads((run_dir / "config.json").read_text())
    assert training_config["attribute_names"] == BENCHMARK["attribute_names"]

    # 使用训练时保存的源码，检查全部快照文件。
    for name, expected_hash in training_config["source_sha256"].items():
        source_path = run_dir / "source" / name
        assert source_path.is_file(), f"Missing snapshot: {source_path}"
        assert sha_file(source_path) == expected_hash, (
            f"Training snapshot changed: {source_path}"
        )

    binding = BENCHMARK["threshold_bindings"][run]
    threshold_path = WORKSPACE / binding["path"]
    assert sha_file(threshold_path) == binding["file_sha256"]
    threshold = json.loads(threshold_path.read_text())
    assert threshold["thresholds"] == binding["thresholds"]
    assert threshold["attribute_names"] == BENCHMARK["attribute_names"]

    checkpoint_path = run_dir / binding["checkpoint"]
    print("Computing checkpoint SHA256...")
    assert sha_file(checkpoint_path) == binding["checkpoint_sha256"]

    # 这些是自己训练并经过哈希验证的 checkpoint。
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    print("Checkpoint keys:", sorted(checkpoint))
    assert checkpoint["epoch"] == int(
        Path(binding["checkpoint"]).stem.split("_")[-1]
    )
    checkpoint_config = checkpoint["config"]
    assert checkpoint_config["attribute_names"] == BENCHMARK["attribute_names"]
    assert (
        checkpoint_config["source_sha256"]
        == training_config["source_sha256"]
    )

    source_path = run_dir / "source" / relative
    spec = importlib.util.spec_from_file_location(
        f"day19_trained_{run}", source_path
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    # 不下载 ImageNet 权重，直接加载选定 checkpoint。
    model = getattr(module, class_name)(
        num_attributes=24, pretrained=False
    )
    state_keys = [
        key for key in ("model_state", "model_state_dict")
        if key in checkpoint
    ]
    assert len(state_keys) == 1, f"Unexpected state keys: {state_keys}"
    model.load_state_dict(checkpoint[state_keys[0]], strict=True)
    model.eval()
    MODELS[run] = model

    MODEL_PROVENANCE[run] = {
        "checkpoint_sha256": binding["checkpoint_sha256"],
        "model_source_sha256": sha_file(source_path),
        "state_key": state_keys[0],
    }
    del checkpoint
    print("Training snapshots, frozen thresholds, strict loading: PASS")

print("\nWorkspace:", WORKSPACE)
print("No model inference or test evaluation performed")
print("Day 19 frozen-input and checkpoint preparation: PASS")

Saving day19_frozen_inputs.zip to day19_frozen_inputs.zip
Frozen ZIP and Day 17 audit: PASS

===== r18_seed0 =====
Computing checkpoint SHA256...
Checkpoint keys: ['config', 'epoch', 'history', 'model_state', 'optimizer_state']
Training snapshots, frozen thresholds, strict loading: PASS

===== r50_seed0 =====
Computing checkpoint SHA256...
Checkpoint keys: ['config', 'epoch', 'history', 'model_state', 'optimizer_state']
Training snapshots, frozen thresholds, strict loading: PASS

===== m3_seed0 =====
Computing checkpoint SHA256...
Checkpoint keys: ['best', 'config', 'epoch', 'history', 'model_state_dict', 'optimizer_state_dict']
Training snapshots, frozen thresholds, strict loading: PASS

Workspace: /content/day19_benchmark_workspace
No model inference or test evaluation performed
Day 19 frozen-input and checkpoint preparation: PASS


In [ ]:
import subprocess
import sys
import importlib.metadata
import importlib.util
import json
import torch

# 与 Day16 相同；保留当前已经可用的 torch / torchvision。
try:
    installed = importlib.metadata.version("facenet-pytorch")
except importlib.metadata.PackageNotFoundError:
    installed = None

if installed != "2.6.0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-deps", "facenet-pytorch==2.6.0",
    ])

assert importlib.metadata.version("facenet-pytorch") == "2.6.0"

quality = json.loads(
    (WORKSPACE /
     "docs/results/corruption_preparation/day16/detector_quality_report.json")
    .read_text()
)

detector_path = WORKSPACE / "detectors/mtcnn.py"
assert sha_file(detector_path) == quality["source_sha256"]["detectors/mtcnn.py"]

spec = importlib.util.spec_from_file_location(
    "day19_frozen_mtcnn", detector_path
)
detector_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(detector_module)
MTCNNDetector = detector_module.MTCNNDetector

assert MTCNNDetector.CONFIG == quality["detector_config"]
assert list(MTCNNDetector.POINT_ORDER) == quality["point_order"]

torch.set_num_threads(quality["torch_threads"])
DETECTOR = MTCNNDetector(device="cpu")

print("facenet-pytorch:", importlib.metadata.version("facenet-pytorch"))
print("Detector source, configuration and point order: PASS")
print("MTCNN initialization: PASS")
print("Detector device:", DETECTOR.device)
print("Torch threads:", torch.get_num_threads())

# 找出旧 notebook 中权重哈希的计算方式，直接复用。
notebook_path = (
    PROJECT / "notebooks/day16_detector_quality_and_pipeline_smoke.ipynb"
)
print("\nDay16 notebook exists:", notebook_path.is_file())

if notebook_path.is_file():
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    print("Existing weight-hash code:")
    for cell_index, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue
        source = cell.get("source", [])
        source = source if isinstance(source, str) else "".join(source)
        lines = source.splitlines()
        selected_lines = set()
        for index, line in enumerate(lines):
            if any(word in line for word in (
                "state_dict(", "detector_weights_sha256",
                "hashlib.", ".tobytes(",
            )):
                selected_lines.update(
                    range(max(0, index - 8), min(len(lines), index + 9))
                )
        if selected_lines:
            print(f"\n--- Cell {cell_index} ---")
            for index in sorted(selected_lines):
                print(f"{index + 1}: {lines[index]}")

print("\nNo test images or model inference used")

facenet-pytorch: 2.6.0
Detector source, configuration and point order: PASS
MTCNN initialization: PASS
Detector device: cpu
Torch threads: 2

Day16 notebook exists: False

No test images or model inference used


In [ ]:
import importlib
import io
import sys
import types
import zipfile
import numpy as np
from PIL import Image

# 用独立包名加载冻结代码，避免与 Colab 已安装的 datasets 包冲突。
package_name = "day19_frozen_datasets"
package = types.ModuleType(package_name)
package.__path__ = [str(WORKSPACE / "datasets")]
sys.modules[package_name] = package

preprocessing = importlib.import_module(
    package_name + ".benchmark_preprocessing"
)
corruptions = importlib.import_module(package_name + ".corruptions")

# 查找已知的数据目录，不扫描所有图片。
data_directories = [
    PROJECT / "data",
    PROJECT / "data/raw",
    PROJECT / "data/celeba",
    PROJECT,
]

def locate_data_file(name):
    matches = sorted({
        directory / name
        for directory in data_directories
        if (directory / name).is_file()
    })
    assert len(matches) == 1, (
        f"{name}: found {matches}; 请贴出此信息后再继续"
    )
    return matches[0]

partition_path = locate_data_file("list_eval_partition.txt")
landmark_path = locate_data_file("list_landmarks_align_celeba.txt")
image_zip_path = locate_data_file("img_align_celeba.zip")

assert sha_file(partition_path) == freeze["partition_sha256"]

sample_names = quality["sample_filenames"]
wanted = set(sample_names)

partition = {}
for line in partition_path.read_text(encoding="utf-8-sig").splitlines():
    if line.strip():
        name, split = line.split()
        if name in wanted:
            partition[name] = split
assert set(partition) == wanted
assert all(split == "1" for split in partition.values())

landmarks = {}
with landmark_path.open(encoding="utf-8-sig") as stream:
    next(stream)
    next(stream)
    for line in stream:
        name, *coordinates = line.split()
        if name in wanted:
            landmarks[name] = torch.tensor(
                [float(value) for value in coordinates],
                dtype=torch.float32,
            )
assert set(landmarks) == wanted

# 只从 ZIP 读取四张 validation 图片，不解压全量数据。
images = {}
with zipfile.ZipFile(image_zip_path) as archive:
    members = {}
    for member in archive.infolist():
        name = member.filename.rsplit("/", 1)[-1]
        if not member.is_dir() and name in wanted:
            assert name not in members, f"Duplicate ZIP image: {name}"
            members[name] = member.filename
    assert set(members) == wanted

    for name in sample_names:
        with Image.open(io.BytesIO(archive.read(members[name]))) as image:
            images[name] = image.convert("RGB")

print("Four official validation images loaded:", sample_names)

canvases = {
    name: preprocessing.prepare_canvas(images[name], landmarks[name])
    for name in sample_names
}

checked = 0
maximum_gt_error = 0.0
maximum_detector_difference = 0.0
status_counts = {}
SMOKE_INPUTS = []

for reference in quality["records"]:
    name = reference["sample_id"]
    condition = reference["condition"]
    severity = reference["severity"]
    canvas, gt = canvases[name]

    final_image, transformed_gt, metadata = corruptions.apply_corruption(
        canvas,
        "blur" if condition == "clean" else condition,
        severity,
        sample_id=name,
        seed=BENCHMARK["corruption_seed"],
        gt_pixels=gt,
    )

    key = f"{name} {condition} s{severity}"
    assert (
        corruptions.rgb_sha256(final_image)
        == reference["input_rgb_sha256"]
    ), f"Final RGB differs: {key}"
    actual_params = metadata["parameters"]
    expected_params = reference["corruption_parameters"]
    assert actual_params.keys() == expected_params.keys(), key

    for field in actual_params:
        if field == "forward_matrix_pixel_coordinates":
            np.testing.assert_allclose(
                actual_params[field],
                expected_params[field],
                rtol=0,
                atol=1e-12,
                err_msg=f"Rotation matrix differs: {key}",
            )
        else:
            assert actual_params[field] == expected_params[field], (
                f"Corruption parameter differs: {key}: {field}"
            )

    gt_error = float(np.max(np.abs(
        np.asarray(transformed_gt)
        - np.asarray(reference["gt_pixels"])
    )))
    maximum_gt_error = max(maximum_gt_error, gt_error)
    assert gt_error <= 1e-6, f"GT differs: {key}: {gt_error}"

    detection = DETECTOR.detect(final_image)
    status = detection["status"]
    assert status == reference["detection"]["status"], (
        f"Detection status differs: {key}: "
        f"{status} vs {reference['detection']['status']}"
    )
    assert (
        detection["face_count"] == reference["detection"]["face_count"]
    ), f"Face count differs: {key}"
    assert (
        detection["selected_index"]
        == reference["detection"]["selected_index"]
    ), f"Selected face differs: {key}"

    valid = status == "ok"
    if valid:
        predicted = np.asarray(detection["landmarks"])
        previous = np.asarray(reference["detection"]["landmarks"])
        maximum_detector_difference = max(
            maximum_detector_difference,
            float(np.max(np.abs(predicted - previous))),
        )
        points = preprocessing.normalize_points(predicted)
    else:
        points = torch.zeros(10, dtype=torch.float32)

    tensor = preprocessing.normalize_canvas(final_image)
    assert tensor.shape == (3, 224, 224)
    assert torch.isfinite(tensor).all()
    assert torch.isfinite(points).all()

    SMOKE_INPUTS.append((tensor, points, valid))
    status_counts[status] = status_counts.get(status, 0) + 1
    checked += 1

assert checked == quality["unique_inputs"] == 84

print("Final RGB hashes, corruption parameters and GT: PASS")
print("Detection status, face count and selection reproduction: PASS")
print("Conditions checked:", checked)
print("Detection status counts:", status_counts)
print("Maximum GT difference:", maximum_gt_error, "px")
print("Maximum detector landmark difference:",
      maximum_detector_difference, "px")
print("Detector coordinate differences are reported separately.")
print("No attribute metrics or test evaluation performed")

Four official validation images loaded: ['162771.jpg', '169393.jpg', '176015.jpg', '182637.jpg']
Final RGB hashes, corruption parameters and GT: PASS
Detection status, face count and selection reproduction: PASS
Conditions checked: 84
Detection status counts: {'ok': 82, 'no_face': 2}
Maximum GT difference: 2.842170943040401e-14 px
Maximum detector landmark difference: 4.57763671875e-05 px
Detector coordinate differences are reported separately.
No attribute metrics or test evaluation performed


In [ ]:
import random
import numpy as np
import torch

assert torch.cuda.is_available()
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

DEVICE = torch.device("cuda")
for model in MODELS.values():
    model.to(DEVICE).eval()

smoke_images = torch.stack([item[0] for item in SMOKE_INPUTS])
smoke_points = torch.stack([item[1] for item in SMOKE_INPUTS])
smoke_valid = torch.tensor(
    [item[2] for item in SMOKE_INPUTS], dtype=torch.bool
)
assert len(smoke_images) == 84
assert int((~smoke_valid).sum()) == 2

SMOKE_LOGITS = {}
with torch.inference_mode():
    for run, model in MODELS.items():
        outputs = []
        for start in range(0, 84, 16):
            stop = min(start + 16, 84)
            image_batch = smoke_images[start:stop].to(DEVICE)

            if run == "m3_seed0":
                logits = model(
                    image_batch,
                    smoke_points[start:stop].to(DEVICE),
                    smoke_valid[start:stop].to(DEVICE),
                )
            else:
                logits = model(image_batch)

            assert logits.shape == (stop - start, 24)
            assert torch.isfinite(logits).all()
            outputs.append(logits.cpu())

        SMOKE_LOGITS[run] = torch.cat(outputs)
        print(run, "finite logits:", tuple(SMOKE_LOGITS[run].shape), "PASS")

    # 同一批失败样本：改变无效坐标，输出应完全不变。
    invalid_images = smoke_images[~smoke_valid].to(DEVICE)
    invalid_mask = torch.zeros(
        len(invalid_images), dtype=torch.bool, device=DEVICE
    )
    zero_points = torch.zeros(
        len(invalid_images), 10, device=DEVICE
    )
    m3 = MODELS["m3_seed0"]
    baseline = m3(invalid_images, zero_points, invalid_mask)

    for name, value in (("large_finite", 1e6), ("nan", float("nan"))):
        changed = m3(
            invalid_images,
            torch.full_like(zero_points, value),
            invalid_mask,
        )
        assert torch.isfinite(changed).all()
        difference = float((changed - baseline).abs().max().item())
        assert difference == 0.0, (name, difference)
        print("M3 invalid-coordinate invariance:", name, difference, "PASS")

print("Day 19 validation forward and mask smoke: PASS")
print("No attribute metrics or test evaluation performed")

r18_seed0 finite logits: (84, 24) PASS
r50_seed0 finite logits: (84, 24) PASS
m3_seed0 finite logits: (84, 24) PASS
M3 invalid-coordinate invariance: large_finite 0.0 PASS
M3 invalid-coordinate invariance: nan 0.0 PASS
Day 19 validation forward and mask smoke: PASS
No attribute metrics or test evaluation performed


In [ ]:
import hashlib

weight_hash = hashlib.sha256()
for name, tensor in sorted(DETECTOR.model.state_dict().items()):
    value = tensor.detach().cpu().contiguous().numpy()
    weight_hash.update(name.encode("utf-8"))
    weight_hash.update(str(value.dtype).encode("ascii"))
    weight_hash.update(str(value.shape).encode("ascii"))
    weight_hash.update(value.tobytes())

DETECTOR_WEIGHTS_SHA256 = weight_hash.hexdigest()
expected = quality["detector_weights_sha256"]

print("Day16 weights:", expected)
print("Current weights:", DETECTOR_WEIGHTS_SHA256)
assert DETECTOR_WEIGHTS_SHA256 == expected, "Detector weights differ"

print("Day16 detector weight reproduction: PASS")

Day16 weights: e9c982616b8d07b3674643a76ce2facd79a2d64fb02b316a827163af2a63d8c0
Current weights: e9c982616b8d07b3674643a76ce2facd79a2d64fb02b316a827163af2a63d8c0
Day16 detector weight reproduction: PASS


In [ ]:
import csv
import importlib.util
import os
import shutil
import sys
import types
import zipfile
from pathlib import Path

DATA_ROOT = Path("/content/day19_benchmark_data/raw")
IMAGE_ROOT = DATA_ROOT / "img_align_celeba"
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

# 核对冻结清单与官方 test 划分。
manifest_path = WORKSPACE / BENCHMARK["official_test_manifest"]
assert sha_file(manifest_path) == BENCHMARK["sample_manifest_sha256"]

with manifest_path.open(newline="") as stream:
    TEST_NAMES = [
        row["sample_id"] for row in csv.DictReader(stream)
    ]

official_test = []
for line in partition_path.read_text(encoding="utf-8-sig").splitlines():
    if line.strip():
        name, split = line.split()
        if split == "2":
            official_test.append(name)

assert TEST_NAMES == sorted(official_test)
assert len(TEST_NAMES) == len(set(TEST_NAMES)) == 19962

DAY19_CONDITIONS = [
    condition for condition in BENCHMARK["conditions"]
    if condition["condition"] in ("clean", "blur", "brightness")
]
assert len(DAY19_CONDITIONS) == 9
print("Official test IDs and frozen order: PASS")
print("Day19 conditions:", len(DAY19_CONDITIONS))

# 复制注释文件；已有文件必须一致。
ANNOTATION_SHA256 = {}
for name in (
    "list_eval_partition.txt",
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
):
    source = locate_data_file(name)
    data = source.read_bytes()
    destination = DATA_ROOT / name
    if destination.exists():
        assert destination.read_bytes() == data, destination
    else:
        destination.write_bytes(data)
    ANNOTATION_SHA256[name] = sha(data)

# 顺序读取 Drive ZIP，并记录整个 ZIP 的哈希。
local_zip = Path("/content/day19_img_align_celeba.zip")
temporary_zip = local_zip.with_suffix(".copying")
digest = hashlib.sha256()
copied = 0
next_progress = 256 * 1024 * 1024

print("Copying image ZIP from Drive...")
with image_zip_path.open("rb") as source, temporary_zip.open("wb") as target:
    while True:
        chunk = source.read(8 * 1024 * 1024)
        if not chunk:
            break
        digest.update(chunk)
        target.write(chunk)
        copied += len(chunk)
        if copied >= next_progress:
            print(f"Copied {copied / 1024**2:.0f} MiB", flush=True)
            next_progress += 256 * 1024 * 1024

IMAGE_ZIP_SHA256 = digest.hexdigest()
if local_zip.exists():
    assert sha_file(local_zip) == IMAGE_ZIP_SHA256, (
        "Existing local ZIP differs; preserved"
    )
    temporary_zip.unlink()
else:
    temporary_zip.replace(local_zip)

# ZIP.read 同时检查各图片成员的 CRC。
wanted = set(TEST_NAMES)
with zipfile.ZipFile(local_zip) as archive:
    members = {}
    for member in archive.infolist():
        name = member.filename.rsplit("/", 1)[-1]
        if not member.is_dir() and name in wanted:
            assert name not in members, f"Duplicate ZIP image: {name}"
            members[name] = member.filename
    assert set(members) == wanted

    for index, name in enumerate(TEST_NAMES, 1):
        data = archive.read(members[name])
        destination = IMAGE_ROOT / name
        if destination.exists():
            assert destination.read_bytes() == data, destination
        else:
            temporary = destination.with_suffix(".jpg.tmp")
            temporary.write_bytes(data)
            temporary.replace(destination)
        if index % 2000 == 0 or index == len(TEST_NAMES):
            print(f"Prepared {index}/{len(TEST_NAMES)}", flush=True)

assert {
    path.name for path in IMAGE_ROOT.iterdir() if path.is_file()
} == wanted

# 直接加载 M3 训练快照中的 Dataset，不重写标签/GT解析。
snapshot = PROJECT / "runs/m3_seed0/source"
dataset_path = snapshot / "datasets/celeba.py"
training_config = json.loads(
    (PROJECT / "runs/m3_seed0/config.json").read_text()
)
assert sha_file(dataset_path) == (
    training_config["source_sha256"]["datasets/celeba.py"]
)

# 只在模块加载期间解析其原有 datasets.preprocessing 导入。
aliases = ("datasets", "datasets.preprocessing")
previous = {name: sys.modules.get(name) for name in aliases}
dataset_package = types.ModuleType("datasets")
dataset_package.__path__ = [str(snapshot / "datasets")]

try:
    sys.modules["datasets"] = dataset_package
    sys.modules["datasets.preprocessing"] = importlib.import_module(
        "day19_frozen_datasets.preprocessing"
    )
    spec = importlib.util.spec_from_file_location(
        "day19_training_celeba", dataset_path
    )
    dataset_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(dataset_module)
finally:
    for name, original in previous.items():
        if original is None:
            sys.modules.pop(name, None)
        else:
            sys.modules[name] = original

previous_directory = Path.cwd()
try:
    os.chdir(WORKSPACE)
    TEST_DATASET = dataset_module.CelebAAttributes(
        split="test", root=DATA_ROOT
    )
finally:
    os.chdir(previous_directory)

assert TEST_DATASET.names == TEST_NAMES
assert TEST_DATASET.attribute_names == BENCHMARK["attribute_names"]
assert len(TEST_DATASET) == 19962

print("\nTest image membership and Dataset order: PASS")
print("Attribute order: PASS")
print("Test images:", len(TEST_DATASET))
print("Unique image-condition inputs:", len(TEST_DATASET) * 9)
print("Three-model sample evaluations:", len(TEST_DATASET) * 9 * 3)
print("Data root:", DATA_ROOT)
print("Image ZIP SHA256:", IMAGE_ZIP_SHA256)
print("No model inference or test metrics computed")
print("Day 19 test-data preparation: PASS")

Official test IDs and frozen order: PASS
Day19 conditions: 9
Copying image ZIP from Drive...
Copied 256 MiB
Copied 512 MiB
Copied 768 MiB
Copied 1024 MiB
Copied 1280 MiB
Prepared 2000/19962
Prepared 4000/19962
Prepared 6000/19962
Prepared 8000/19962
Prepared 10000/19962
Prepared 12000/19962
Prepared 14000/19962
Prepared 16000/19962
Prepared 18000/19962
Prepared 19962/19962

Test image membership and Dataset order: PASS
Attribute order: PASS
Test images: 19962
Unique image-condition inputs: 179658
Three-model sample evaluations: 538974
Data root: /content/day19_benchmark_data/raw
Image ZIP SHA256: 46fb89443c578308acf364d7d379fe1b9efb793042c0af734b6112e4fd3a8c74
No model inference or test metrics computed
Day 19 test-data preparation: PASS


In [ ]:
from google.colab import files
import importlib.util
from pathlib import Path

expected_runner_sha256 = (
    "330b07cbe05f71df8a7266422222263c816d4e63751df3c8ca4aa5e2c232f228"
)
runner_path = WORKSPACE / "scripts/day19_evaluate_benchmark.py"

if not runner_path.exists():
    uploaded_runner = files.upload()
    assert len(uploaded_runner) == 1, "请只上传评估脚本"
    runner_bytes = next(iter(uploaded_runner.values()))
    assert sha(runner_bytes) == expected_runner_sha256
    runner_path.parent.mkdir(parents=True, exist_ok=True)
    runner_path.write_bytes(runner_bytes)

assert sha_file(runner_path) == expected_runner_sha256

spec = importlib.util.spec_from_file_location("day19_runner", runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

# 保存执行脚本和冻结文件，便于追溯和恢复。
source_archive = PROJECT / "runs/benchmark/day19/source"
runner.write_once(
    source_archive / "scripts/day19_evaluate_benchmark.py",
    runner_path.read_bytes(),
)
handoff = json.loads(
    (WORKSPACE / "day19_handoff_manifest.json").read_text()
)
for name, expected_hash in handoff["files"].items():
    data = (WORKSPACE / name).read_bytes()
    assert sha(data) == expected_hash
    runner.write_once(source_archive / "frozen_inputs" / name, data)
runner.write_once(
    source_archive / "frozen_inputs/day19_handoff_manifest.json",
    (WORKSPACE / "day19_handoff_manifest.json").read_bytes(),
)

DAY19_REPORT = runner.run_day19(
    workspace=WORKSPACE,
    project=PROJECT,
    dataset=TEST_DATASET,
    models=MODELS,
    detector=DETECTOR,
    preprocessing=preprocessing,
    corruptions=corruptions,
    model_provenance=MODEL_PROVENANCE,
    detector_weights_sha256=DETECTOR_WEIGHTS_SHA256,
    annotation_sha256=ANNOTATION_SHA256,
    image_zip_sha256=IMAGE_ZIP_SHA256,
)

Saving day19_evaluate_benchmark.py to day19_evaluate_benchmark.py
Binding test JPEG bytes...
Official test inference starts: 9 conditions, 19962 images, 3 models

=== clean_s0 ===
clean_s0: 256/19962; saved; elapsed=0.2 min; remaining estimate=152.7 min
clean_s0: 512/19962; saved; elapsed=0.4 min; remaining estimate=151.6 min
clean_s0: 768/19962; saved; elapsed=0.7 min; remaining estimate=151.6 min
clean_s0: 1024/19962; saved; elapsed=0.9 min; remaining estimate=151.3 min
clean_s0: 1280/19962; saved; elapsed=1.1 min; remaining estimate=151.0 min
clean_s0: 1536/19962; saved; elapsed=1.3 min; remaining estimate=150.7 min
clean_s0: 1792/19962; saved; elapsed=1.5 min; remaining estimate=150.4 min
clean_s0: 2048/19962; saved; elapsed=1.7 min; remaining estimate=150.2 min
clean_s0: 2304/19962; saved; elapsed=1.9 min; remaining estimate=149.6 min
clean_s0: 2560/19962; saved; elapsed=2.2 min; remaining estimate=148.9 min
clean_s0: 2816/19962; saved; elapsed=2.4 min; remaining estimate=148.6 mi

In [1]:
from pathlib import Path
import json

drive_mount = Path("/content/drive/MyDrive")
if not drive_mount.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

output = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/benchmark/day19"
)

required = [
    "WORKSPACE", "PROJECT", "TEST_DATASET", "MODELS",
    "DETECTOR", "preprocessing", "corruptions",
    "MODEL_PROVENANCE", "DETECTOR_WEIGHTS_SHA256",
    "ANNOTATION_SHA256", "IMAGE_ZIP_SHA256",
]
missing = [name for name in required if name not in globals()]
print("Missing runtime variables:", missing)

print("\nSaved batches on Drive:")
for kind, severities in (
    ("clean", [0]),
    ("blur", range(1, 5)),
    ("brightness", range(1, 5)),
):
    for severity in severities:
        label = f"{kind}_s{severity}"
        shards = sorted((output / "shards" / label).glob("*.zip"))
        contiguous_end = 0
        for shard in shards:
            start, stop = map(int, shard.stem.split("_"))
            if start != contiguous_end:
                break
            contiguous_end = stop
        print(f"{label}: {len(shards)} files; "
              f"continuous saved range ends at {contiguous_end}/19962")

report_path = output / "day19_benchmark_report.json"
print("\nFinal report exists:", report_path.is_file())
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Day19 complete:", report["day19_scope_complete"])

print("以上只检查文件是否存在；续跑时脚本会验证每批内容和哈希。")

Mounted at /content/drive
Missing runtime variables: ['WORKSPACE', 'PROJECT', 'TEST_DATASET', 'MODELS', 'DETECTOR', 'preprocessing', 'corruptions', 'MODEL_PROVENANCE', 'DETECTOR_WEIGHTS_SHA256', 'ANNOTATION_SHA256', 'IMAGE_ZIP_SHA256']

Saved batches on Drive:
clean_s0: 78 files; continuous saved range ends at 19962/19962
blur_s1: 78 files; continuous saved range ends at 19962/19962
blur_s2: 78 files; continuous saved range ends at 19962/19962
blur_s3: 48 files; continuous saved range ends at 12288/19962
blur_s4: 0 files; continuous saved range ends at 0/19962
brightness_s1: 0 files; continuous saved range ends at 0/19962
brightness_s2: 0 files; continuous saved range ends at 0/19962
brightness_s3: 0 files; continuous saved range ends at 0/19962
brightness_s4: 0 files; continuous saved range ends at 0/19962

Final report exists: False
以上只检查文件是否存在；续跑时脚本会验证每批内容和哈希。


In [2]:
from google.colab import files
from pathlib import Path
import hashlib
import importlib.util

uploaded = files.upload()
assert len(uploaded) == 1, "请只上传 day19_resume.py"
data = next(iter(uploaded.values()))

assert hashlib.sha256(data).hexdigest() == (
    "5357d44f9781052e4a262b8cd8ac7f67ccb0354cc6936401bd9a7a0013eba495"
)

path = Path("/content/day19_resume.py")
path.write_bytes(data)

spec = importlib.util.spec_from_file_location("day19_recovery", path)
recovery = importlib.util.module_from_spec(spec)
spec.loader.exec_module(recovery)

DAY19_REPORT = recovery.restore_and_resume()

Saving day19_resume.py to day19_resume.py
Original execution environment: MATCH
Restoring r18_seed0: verifying snapshot and checkpoint...
r18_seed0: strict restoration PASS
Restoring r50_seed0: verifying snapshot and checkpoint...
r50_seed0: strict restoration PASS
Restoring m3_seed0: verifying snapshot and checkpoint...
m3_seed0: strict restoration PASS
Original detector weights: MATCH
Restoring local image ZIP from Drive...
Copied 256 MiB
Copied 512 MiB
Copied 768 MiB
Copied 1024 MiB
Copied 1280 MiB
Restored test JPEGs: 2000/19962
Restored test JPEGs: 4000/19962
Restored test JPEGs: 6000/19962
Restored test JPEGs: 8000/19962
Restored test JPEGs: 10000/19962
Restored test JPEGs: 12000/19962
Restored test JPEGs: 14000/19962
Restored test JPEGs: 16000/19962
Restored test JPEGs: 18000/19962
Restored test JPEGs: 19962/19962
Original 19962 test JPEG hashes: MATCH
Day19 runtime restoration: PASS; resuming original runner
Binding test JPEG bytes...
Official test inference starts: 9 condition

In [3]:
from pathlib import Path
import hashlib
import json
import zipfile
from google.colab import files

root = Path("/content/drive/MyDrive/FacialVisualProfile/runs/benchmark/day19")

def sha_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

report_path = root / "day19_benchmark_report.json"
report = json.loads(report_path.read_text())
assert report["day19_scope_complete"] is True
assert report["full_benchmark_complete"] is False
assert report["threshold_fitting_performed"] is False
assert report["runtime_errors"] == 0
assert report["condition_count"] == 9
assert report["test_sample_count"] == 19962

assert sha_file(root / "execution_binding.json") == (
    report["execution_binding_sha256"]
)

# 核对最终报告引用的所有预测批次。
assert len(report["shard_sha256"]) == 9 * 78
for name, expected in report["shard_sha256"].items():
    relative = Path(name)
    assert not relative.is_absolute() and ".." not in relative.parts
    assert sha_file(root / relative) == expected, name

payloads = {
    name: (root / name).read_bytes()
    for name in (
        "day19_benchmark_report.json",
        "execution_binding.json",
        "test_jpeg_sha256.json",
    )
}

for path in sorted((root / "condition_reports").glob("*.json")):
    payloads["condition_reports/" + path.name] = path.read_bytes()

runner_path = root / "source/scripts/day19_evaluate_benchmark.py"
binding = json.loads(payloads["execution_binding.json"])
assert sha_file(runner_path) == binding["runner_sha256"]
payloads["scripts/day19_evaluate_benchmark.py"] = runner_path.read_bytes()

resume_path = Path("/content/day19_resume.py")
assert sha_file(resume_path) == (
    "5357d44f9781052e4a262b8cd8ac7f67ccb0354cc6936401bd9a7a0013eba495"
)
payloads["scripts/day19_resume.py"] = resume_path.read_bytes()

manifest = {
    "day19_scope_complete": True,
    "verified_prediction_shards": len(report["shard_sha256"]),
    "prediction_shards_location": str(root / "shards"),
    "files": {
        name: hashlib.sha256(data).hexdigest()
        for name, data in payloads.items()
    },
}
payloads["archive_manifest.json"] = (
    json.dumps(manifest, indent=2) + "\n"
).encode()

destination = Path("/content/day19_results_archive.zip")
with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
    for name, data in sorted(payloads.items()):
        archive.writestr(name, data)

print("Verified prediction shards:", len(report["shard_sha256"]))
print("Day19 result archive: PASS")
files.download(str(destination))

Verified prediction shards: 702
Day19 result archive: PASS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>